<a href="https://colab.research.google.com/github/ds-20195/notebooks/blob/main/drafts/Notebook2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data and the State (DATA 21905/31905)
# Lab 2: The Decennial Census (blocks) and the ACS

In Lab 1, we explored the most famous and essential dataset published by the U.S. Census Bureau at the **county** level in Illinois. In this lab we go deeper in two directions:

1. **Finer geography:** from counties down to individual census **blocks**, the smallest unit in the Census hierarchy.
2. **Richer data:** from the decennial Census to the **American Community Survey (ACS)**, which covers income, housing, commuting, education, and much more.

By the end, we'll combine race data with socioeconomic variables to ask questions about neighborhood composition — and run into a classic statistical pitfall along the way.

In [ ]:
!pip install -q census us

In [ ]:
%config InlineBackend.figure_formats = ["svg"]

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from census import Census
from us import states

state_fips  = "17"    # Illinois
county_fips = "031"   # Cook County
state_name  = "Illinois"
county_name = "Cook County"

census = Census("", year=2020)

In [ ]:
# @title Check the census API is working
try:
    county_populations = census.pl.get(
        ("NAME"),
        geo={"for": "state:17"}
    )
except Exception:
    raise RuntimeError(
        "Error fetching data from the Census API. "
        "Please try disconnecting and reconnecting the runtime. "
        "If the problem persists after 2 resets, you will need to use our backup data cells to "
        "complete the lab"
    )

## Section 1 — The Central Spine

## Section 2 — GEOIDs and Block Level Data

Census **blocks** are the smallest unit in the hierarchy. While counties vary widely in population and tracts are designed for comparability, blocks follow physical features — streets, waterways, property lines. Some blocks have hundreds of people; many have zero.

Here we'll fetch the same P1 race table from Lab 1, but at block resolution for Cook County. The API call structure is identical; we just change `"for": "county:*"` to `"for": "block:*"` and specify the county. **Cook County has roughly 43,000 blocks**, so this could take up to a minute.

In [ ]:
# We will repeat some of the work from Notebook 1 here
p1_population_columns = {
    "P1_003N": "White",     # White alone
    "P1_004N": "Black",     # Black or African American alone
    "P1_005N": "AMIN",      # American Indian and Alaska Native alone
    "P1_006N": "Asian",     # Asian alone
    "P1_007N": "NHPI",      # Native Hawaiian and Other Pacific Islander alone
    "P1_008N": "SomeOther", # Some Other Race alone
    "P1_009N": "Multi",     # Population of two or more races
}
pl_race_categories = list(p1_population_columns.values())

block_populations = census.pl.get(
    ("NAME", *p1_population_columns),
    geo={
        "for": "block:*",   # Note: we changed the "for" statement to include blocks
        "in": f"county:{county_fips} state:{state_fips}", # We now modify the "in" statment to include
                                                          # both the county and state FIPS codes
    },
)

print(f"Fetched data for {len(block_populations):,} blocks.")

# now let's make sure we're using human-readable columns, as chosen above
block_race_df = pd.DataFrame(block_populations).rename(
    columns={"NAME": "name", **p1_population_columns}
)

We'll dive in and work with block data.  Let's introduce the [GEOID](https://www.census.gov/programs-surveys/geography/guidance/geo-identifiers.html):  a numeric code the US Census Bureau uses to name their geographies.  It starts with 2 digits for the state, which is the FIPS code we already saw last time.  Then 3 digits for the county, 6 for the tract, 1 for the block group, and 3 more for the block.  (You can think of that together as four for the block.)

So for instance 482012231001050 is a block in Texas (state 48), particularly Harris County (201), tract 2231 (223100), block group 1, which is incorporated into the block code 1050.  The nice thing about this naming scheme is that you can truncate a block code to get the codes for the higher-level geounits along the spine.

GEOID20 just means the GEOID from the 2020 Census.

In [ ]:
# this constructs the 15-digit complete GEOID by concatenating the pieces
# describing different hierarchical levels: state(2) + county(3) + tract(6) + block(4)
block_race_df["GEOID20"] = (
    block_race_df["state"]
    + block_race_df["county"]
    + block_race_df["tract"]
    + block_race_df["block"]
)

block_race_df

In [ ]:
# Since GEOID is a unique identifier for each block, it is a good choice as an index for our dataframe.
block_race_df = block_race_df.set_index("GEOID20")

# ⚠️ NEW SYNTAX: We have encoded all of the information that is present in the columns
# "name", "state", "county", "tract", and "block" into the GEOID20 column, so we can drop
# those extra columns.
block_race_df = block_race_df.drop(
    columns=["name", "state", "county", "tract", "block"]
)

# Now we convert the population columns to integers rather than strings, which is good data practice
# so we can do integer operations on them, like summing.
block_race_df[pl_race_categories] = block_race_df[pl_race_categories].astype(int)

block_race_df["total"] = block_race_df[pl_race_categories].sum(axis=1)
block_race_df

In [ ]:
# Now we can compute the percentage of each racial category for each block, being careful to
# handle zero division errors by filling NaN values with 0.
for col in pl_race_categories:
    block_race_df[f"{col}_pct"] = (
        100 * block_race_df[col] / block_race_df["total"]
    ).fillna(0) # <- this fills any NaN values with 0, which can happen when the total population is 0

block_race_df

That dataframe view shows that some blocks have zero people, while others have hundreds. Let's get a sense of the largest and smallest.

In [ ]:
# sorts by total population, largest to smallest
block_race_df.sort_values("total", ascending=False)

#### TASK 1

By inspecting the above view, you can find the GEOID of the most populous block in Cook County and figure out its population.

Using boolean filtering, find all blocks in Cook County that are at least 50% Asian. (Think about whether it makes sense to include zero-pop blocks.)

In [ ]:
# YOUR CODE HERE

# raise your hand and ask for help if you get stuck!

#### TASK 2

How many census blocks in Cook County have more than 75% of their population made up of a single Census race?

In [ ]:
# YOUR CODE HERE

# raise your hand and ask for help if you get stuck!

### Population Distribution and Power Laws



Let's check out the distribution of TOTPOP of census blocks in Cook County.

In [ ]:
# histogram of block *populations* — how many people live in each block?
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(block_race_df["total"], bins=40)
ax.set_xlabel("Block population")
ax.set_ylabel("Count")
ax.set_title(f"Distribution of block populations in {county_name}")
plt.show()

That viz is pretty lousy, for a few reasons. Discuss!

We'll walk through a few alternatives just to give you ideas.  A core skill you are trying to build is making good viz, and there's a lot of data-common-sense that goes into that.

In [ ]:
# a version that clips both axes
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(block_race_df["total"], bins=range(0, 510, 10))
ax.set_xlim(0, 500)
ax.set_ylim(0, 5000)
ax.set_xlabel("Block population")
ax.set_ylabel("Count")
ax.set_title(f"How many blocks in each population range in {county_name} (axes clipped)")
plt.show()

Most blocks have very small populations — many are zero. A few are very large. This kind of highly right-skewed distribution is typical of lots of statistics -- not just population of Census blocks.  This kind of shape is called a [power law](https://en.wikipedia.org/wiki/Power_law).

In [ ]:
# find the max so we can take that into account when plotting
block_race_df["total"].max()

In [ ]:
# let's switch to a log scale instead
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(block_race_df["total"], bins=range(0, 4800, 100)) # play around with bin size!!
ax.set_yscale("log")
ax.set_xlabel("Block population")
ax.set_ylabel("Count (log scale)")
ax.set_title(f"Distribution of block populations in {county_name} (log scale)")
plt.show()

We'll start to see some different characteristic patterns when we fold in race.

In [ ]:
# This cell gets county-level population data for IL and saves to `county_race_df`

county_populations = census.pl.get(
    ("NAME", *p1_population_columns),
    geo={"for": "county:*", "in": f"state:{state_fips}"},
)
county_race_df = (
    pd.DataFrame(county_populations)
    .rename(columns={"NAME": "name", **p1_population_columns})
    .set_index("name")
    .drop(columns=["state", "county"])
)
county_race_df[pl_race_categories] = county_race_df[pl_race_categories].astype(int)
county_race_df["total"] = county_race_df[pl_race_categories].sum(axis=1)
for col in pl_race_categories:
    county_race_df[f"{col}_pct"] = (100 * county_race_df[col] / county_race_df["total"]).fillna(0)

print("First 5 rows of 'county_race_df':")
county_race_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(county_race_df["Black_pct"], bins=20)
ax.set_xlabel("% Black")
ax.set_ylabel("Count")
ax.set_title(f"Black population % across {state_name} counties")
plt.show()

In [ ]:
# ⚠️ NEW SYNTAX: When filtering, you can use the notation `df.loc[mask]`. This is the most robust way to
# filter a dataframe when queries get complicated, but you will also see the syntax `df[mask]` used frequently.
populated_blocks_df = block_race_df[block_race_df["total"] > 0]

fig, ax = plt.subplots(figsize=(10, 5))

# histogram of Black% across populated blocks
ax.hist(populated_blocks_df["Black_pct"], bins=20)
ax.set_xlabel("% Black")
ax.set_ylabel("Count")
ax.set_title(f"Black population % across {county_name} blocks")
plt.show()

Notice the shape: it's **bimodal** (having two peaks), with a large spike near 0% and another peak near 100%. This is a quantitative signature of residential segregation. (Very different than the power law shape.)  We'll talk more about measuring segregation in future labs.

## Section 3 — The American Community Survey (ACS)

In addition to publishing decennial data, the U.S. Census Bureau publishes [American Community Survey](https://www.census.gov/programs-surveys/acs) (ACS) data every year. The ACS is sent to a random sample of U.S. addresses; it is an invaluable source for understanding economic and social trends. The key tradeoffs versus the decennial:

| | Decennial Census | ACS 5-year estimates |
|---|---|---|
| Coverage | Full count | ~3.5M households/year (sample) |
| Frequency | Every 10 years | Annually (pooled over 5 years) |
| Variables | Demographics only | Demographics + socioeconomics |
| Smallest unit | Block | Tract (usually) |
| Uncertainty | Exact counts | Has margins of error |


The ACS has much richer socio-economic variables than the Decennial data. We'll start by making a tract-level dataframe that includes age, income, and commute data.

*Note:* The basic way to explore ACS variables is to look at the associated documentation. Specifically, you can find relevant variables and table IDs by looking at the ["ACS Detailed Table Shells"](https://www.census.gov/programs-surveys/acs/technical-documentation/table-shells.2023.html#list-tab-79594641) located on the Census website. Additionally, the course staff put together a [Google sheets workbook](https://docs.google.com/spreadsheets/d/1DtGNarbQLaJdtMiINQ7brQ-Y6zawBkkBkp0VGSENsZw/edit?usp=sharing) that organizes this information for the 2023–2015 ACS and the 2020 and 2010 Decennial Census.  This will be a handy reference!

In [ ]:
# Estimate!!Median age --!!Total:
median_age_column = "B01002_001E"

# Estimate!!Median household income in the past 12 months:
median_income_column = "B19013_001E"

# Means of Transportation to Work: total commuters, and public transit users
commuters_total_column = "B08301_001E"
transit_column         = "B08301_010E"

In [ ]:
county_tract_acs_df = pd.DataFrame(
    census.acs5.get(
        (median_age_column, median_income_column, commuters_total_column, transit_column),
        geo={
            "for": "tract:*",  # Note: we changed the "for" statement to include tracts instead of blocks
            "in": f"county:{county_fips} state:{state_fips}",
        },
        year=2023,
    )
)

# Let's go ahead and rename the columns to something more human-readable
county_tract_acs_df = county_tract_acs_df.rename(columns={
    median_age_column:      "median_age",
    median_income_column:   "median_income",
    commuters_total_column: "commuters_total",
    transit_column:         "transit_riders",
})


# Construct the GEOID column
county_tract_acs_df["GEOID"] = (
    county_tract_acs_df["state"]
    + county_tract_acs_df["county"]
    + county_tract_acs_df["tract"]
)

county_tract_acs_df = county_tract_acs_df.set_index("GEOID")

# We have encoded all of the information that is present in the columns
# "state", "county", and "tract" into the GEOID, so we can drop those columns now.
county_tract_acs_df = county_tract_acs_df.drop(columns=["state", "county", "tract"])


county_tract_acs_df

Something very strange has happened!  There are tracts with negative values in the millions. This is sometimes called a **sentinel value** — dummy data used to flag when something doesn't fit. (It's an alternative to `NaN`, or "not a number.") The Census uses large negative numbers to mean "estimate not available for this tract."

In [ ]:
# how many of these are negative?
print("median age negative in",    (county_tract_acs_df["median_age"] < 0).sum(),    "tracts")
print("median income negative in", (county_tract_acs_df["median_income"] < 0).sum(), "tracts")

# replace sentinel values with NaN
sentinal_value_mask = county_tract_acs_df < 0
county_tract_acs_df[sentinal_value_mask] = np.nan

# derived variable: transit commute share
county_tract_acs_df["transit_pct"] = (
    100 * county_tract_acs_df["transit_riders"] / county_tract_acs_df["commuters_total"]
)

# Sort by transit commute share, largest to smallest
county_tract_acs_df = county_tract_acs_df.sort_values("transit_pct", ascending=False)
county_tract_acs_df

### A first look at the data

Armed with fresh data, let's visualize the distributions of income and age across Cook County tracts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

county_tract_acs_df["median_income"].dropna().hist(bins=20, ax=axes[0])
axes[0].set_xlabel("Median household income ($)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Median income distribution — {county_name} tracts")

county_tract_acs_df["median_age"].dropna().hist(bins=20, ax=axes[1])
axes[1].set_xlabel("Median age (years)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Median age distribution — {county_name} tracts")

plt.tight_layout()
plt.show()

We can also look at which tracts have the highest public transit commute share — a natural variable of interest in a city like Chicago.

In [ ]:
top_transit = (
    county_tract_acs_df["transit_pct"]
    .dropna()
    .sort_values(ascending=False)
    .head(15)
)
ax = top_transit.plot.bar(figsize=(12, 4), color="steelblue")
ax.set_xlabel("Tract GEOID")
ax.set_ylabel("Transit commute share (%)")
ax.set_title(f"Top 15 {county_name} tracts by public transit commute share")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

While it does have interesting info, this not a very informative viz!  Discuss how you might improve it.

Next, let's peek at occupational variables!

In [ ]:
occupation_by_median_earnings_columns = {
    "B24011_002E": "Mgmt/Science/Arts",             # Management, business, science, and arts occupations
    "B24011_018E": "Service",                       # Service occupations
    "B24011_026E": "Sales/Office",                  # Sales and office occupations
    "B24011_029E": "Natural Res./Constr./Maint.",   # Natural resources, construction, and maintenance occupations
    "B24011_033E": "Prod./Transport/Material Mvmt", # Production, transportation, and material moving occupations
}

# Now we do the standard rigmarole to fetch the data, rename columns, construct the GEOID,
# and filter out sentinel values

occupation_columns = list(occupation_by_median_earnings_columns.keys())

county_tract_acs_occupation_df = pd.DataFrame(
    census.acs5.get(
        occupation_columns,
        geo={
            "for": "tract:*",  # Note: we changed the "for" statement to include tracts instead of blocks
            "in": f"county:{county_fips} state:{state_fips}",
        },
        year=2023,
    )
)
county_tract_acs_occupation_df = county_tract_acs_occupation_df.rename(columns=occupation_by_median_earnings_columns)

county_tract_acs_occupation_df["GEOID"] = (
    county_tract_acs_occupation_df["state"]
    + county_tract_acs_occupation_df["county"]
    + county_tract_acs_occupation_df["tract"]
)
county_tract_acs_occupation_df = county_tract_acs_occupation_df.set_index("GEOID")
county_tract_acs_occupation_df = county_tract_acs_occupation_df.drop(columns=["state", "county", "tract"])

sentinal_value_mask = county_tract_acs_occupation_df < 0
county_tract_acs_occupation_df[sentinal_value_mask] = np.nan


county_tract_acs_occupation_df

Let's meet a new kind of plot called a boxplot or box-and-whiskers plot. Is it a good choice for this data?  

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
county_tract_acs_occupation_df.boxplot(
    ax=ax,
    rot=35, # rotate x-axis labels for readability
    whis=(1,99) # set whiskers to 1st and 99th percentiles
)

# turn off gridlines for better readability
ax.grid(False)

# adjust x-axis label alignment and rotation
for label in ax.get_xticklabels():
    label.set_ha("right")
    label.set_rotation_mode("anchor")

plt.show()

In [ ]:
income_by_degree_columns = {
    "B20004_002E": "No degree",
    "B20004_003E": "High school diploma",
    "B20004_004E": "Associate's",
    "B20004_005E": "Bachelor's",
    "B20004_006E": "Master's",
    "B20004_007E": "Graduate / Professional",
}


education_columns = list(income_by_degree_columns.keys())

county_acs_education_df = pd.DataFrame(
    census.acs5.get(
        ["NAME"] + education_columns,
        geo={
            "for": "county:*",
            "in": f"state:{state_fips}",
        },
        year=2023,
    )
)
county_acs_education_df = county_acs_education_df.rename(columns=income_by_degree_columns)

county_acs_education_df["GEOID"] = (
    county_acs_education_df["state"]
    + county_acs_education_df["county"]
)
county_acs_education_df = county_acs_education_df.set_index("GEOID")
county_acs_education_df = county_acs_education_df.drop(columns=["state", "county"])

sentinal_value_mask = county_acs_education_df[list(income_by_degree_columns.values())] < 0
county_acs_education_df[sentinal_value_mask] = np.nan

county_acs_education_df

Matplotlib does not have a built in function for making a heatmap, but we can use the `imshow` function to create one. This takes a 2D array of values and creates a grid of colored squares where the color of each square corresponds to the value in  the array. This is more commonly used for visualizing images (the colors are for pixels), hence the name.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 24))

# imshow
image_object = ax.imshow(
    pd.DataFrame(county_acs_education_df[list(income_by_degree_columns.values())]),
    aspect="auto",
    cmap="viridis",
    vmin=20000,
)


# Now we need to adjust the tick labels to show the degree categories and county names
# instead of numbers. Don't worry about this part: it's a bit of a pain to get right but
# it's mostly just fiddly formatting code.
ax.set_xticks(range(len(income_by_degree_columns))) # make one tick for each degree category
ax.xaxis.tick_top() # move x-axis ticks to the top of the plot
ax.xaxis.set_label_position("top") # move x-axis label to the top as well

# Adjust x-axis tick labels to show degree categories
ax.set_xticklabels(
    list(income_by_degree_columns.values()),
    rotation=30,
    ha="left",  # adjust horizontal alignment of x-axis labels for better readability
)

ax.set_yticks(range(len(county_acs_education_df)))
ax.set_yticklabels(
    # We need to add the '.str' to tell pandas to apply the string replacement to each value in
    # the "NAME" column of the dataframe.
    county_acs_education_df["NAME"].str.replace("County, Illinois", "")
)

ax.set_title(
    f"Illinois Counties: Median Earnings by Educational Attainment\nACS 5-Year ({2023})",
    pad=5,
)

# Add in the color bar
fig.colorbar(image_object, ax=ax)

plt.show()

#### TASK 3

Which tract in Cook County has the **highest median household income**? Which has the **lowest** (ignoring missing values)?

While you're at it: look up the GEOID of the highest-income tract — can you figure out what neighborhood it corresponds to?

In [ ]:
# YOUR CODE HERE

# raise your hand and ask for help if you get stuck!

#### TASK 4

Make a **scatter plot** with median household income on the x-axis and median age on the y-axis, one point per tract. Label the axes and give the plot a title. What pattern do you see?

In [ ]:
# YOUR CODE HERE

# raise your hand and ask for help if you get stuck!

## Section 4 — Joining Decennial and ACS data

Both datasets use GEOID codes as identifiers, but at different scales:
- **Block GEOID**: 15 digits — state(2) + county(3) + tract(6) + block(4)
- **Tract GEOID**: 11 digits — state(2) + county(3) + tract(6)

Since both block and tract are on the central spine of the Census geographic hierarchy, we can get the tract GEOID for any block by taking the first 11 characters of its block GEOID. Then we aggregate block counts up to the tract level and join with the ACS data.

In [ ]:
# To get the tract GEOID, we take the first 11 characters of the full 15-digit block identifier
# Our block dataframe has the GEOIDs already constructed, so let's look at them for a second
block_race_df.index

The important thing to observe here is that the index column is of type "str" which is short for
"string" (as you probably guessed). Like how we can perform math operations (e.g. `sum`, `mean`) on
numeric columns, we can perform string operations on string columns. In particular, we can slice
the string column.

In [ ]:
# This will give us the first 2 characters of the GEOID, which correspond to the state FIPS code.
# Note that we need to use the .str accessor to apply string slicing to every item in the index;
# if we instead tried block_race_df.index[:2] we would just get the first 2 items in the index,
# which is not what we want.
block_race_df.index.str[:2]

In [ ]:
# to get the tract GEOID, we take the first 11 characters of the full 15-digit block identifier
block_race_df["tract_GEOID"] = block_race_df.index.str[:11]
block_race_df

Now that we have the tracts identify, we need to aggregate the data in `block_race_df` up to the
tract level. We do this using the `group_by` method for Pandas dataframes.

In [ ]:
# now we'll group the block data by tract, and aggregate each column by summing over the blocks
# contained in each tract
county_tract_race_df = block_race_df.groupby("tract_GEOID").aggregate("sum")
county_tract_race_df

Recompute the percents...

In [ ]:
# Let's only keep the main pl_race_categories and then recompute

county_tract_race_df = county_tract_race_df[pl_race_categories + ["total"]]
for col in pl_race_categories:
    county_tract_race_df[f"{col}_pct"] = (
        100 * county_tract_race_df[col] / county_tract_race_df["total"]
    ).fillna(0)

# Let's sort by total population really quick to check for NaNs just in case
county_tract_race_df = county_tract_race_df.sort_values("total")
county_tract_race_df

Okay, now we would like to combine our two datasets. To do this we need to make sure that the unique identifiers – GEOID for `county_tract_acs_df` and tract_GEOID for `county_tract_race_df` – line up with each other.


Why is this important? Because pandas is not very good at protecting you from joining data in uninterpretable ways.
Consider the following example:

In [ ]:
fake_df_1 = pd.DataFrame({
    "median_age": [35],
    "GEOID": ["17031010100"]
})

fake_df_2 = pd.DataFrame({
    "median_income": [100],
    "GEOID": ["12"]
})

fake_df_1["median_income"] = fake_df_2["median_income"]
fake_df_1

Uh oh. The data that in `fake_df_2` clearly doesn't belong with the data in `fake_df_1`, but
pandas doesn't know that, and since both `fake_df_1` and `fake_df_2` use the default indexing scheme,
we are able to join data that should not be joined.

Let's try to handle this by setting the indexes to meaningful identifiers.

In [ ]:
fake_df_1 = pd.DataFrame({
    "median_age": [35],
    "GEOID": ["17031010100"]
})
fake_df_1 = fake_df_1.set_index("GEOID")

fake_df_2 = pd.DataFrame({
    "median_income": [100],
    "GEOID": ["12"]
})
fake_df_2 = fake_df_2.set_index("GEOID")

fake_df_1["median_income"] = fake_df_2["median_income"]
fake_df_1

This is better since we don't accidentally put incorrect data into the dataframe, but it would be best
to avoid these misaligned data joins all together. That means that we need to make sure that our index columns
for the two dataframes we wish to join match.

So, we are now going to carefully check and make sure that we can join our `county_tract_race_df` and
our `county_tract_acs_df`.

In [ ]:
# Check that your indices are both unique
print(county_tract_race_df.index.is_unique, county_tract_acs_df.index.is_unique)
# Check that they have the same set of GEOIDs. When the indices are unique, it's easiest to just
# compare the sets of index values directly.
print(set(county_tract_race_df.index) == set(county_tract_acs_df.index))

In [ ]:
# We are now ready to join the dataframes together! Since they both have the same index (tract GEOID),
# we can use panda's built in join function to combine the dataframes together. By default, pandas
# will try to use the indices of the joined dataframes to match rows together. That is why
# it's so important that the indices are unique and that they match across the two dataframes.
combined_df = county_tract_race_df.join(county_tract_acs_df)
combined_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    combined_df["Black_pct"],
    combined_df["median_income"],
    alpha=0.4,
    s=15,
    color="steelblue",
)
ax.set_xlabel("Black population % (2020 Decennial Census)")
ax.set_ylabel("Median household income, $ (2023 ACS 5-year)")
ax.set_title(f"{county_name}: Black population share vs. income by census tract")
plt.tight_layout()
plt.show()

There's a strong pattern here. But be careful about what conclusions you draw: we're looking at **tracts**, not individuals. The correlation at the aggregate level doesn't describe any individual's experience — a Black person in a high-income tract is not evidence against the pattern, and vice versa. This kind of misreading is called the [ecological fallacy](https://en.wikipedia.org/wiki/Ecological_fallacy), and it's a recurring hazard in data analysis with administrative geographies.

#### TASK 5

Choose a different racial group from `pl_race_categories`, or a different ACS variable from `county_tract_acs_df`, and make your own scatter plot. Describe what you see in a sentence or two.

In [ ]:
# YOUR CODE HERE

# raise your hand and ask for help if you get stuck!

## Homework 2

**Warmup question:** Among all Cook County census tracts with income data available, how many have a median household income above $100,000? What fraction of all tracts with data does that represent? Show your pandas code.

**Data product:** Choose a state other than Illinois. Fetch at least one ACS variable at the county or tract level and make at least one visualization.

_Tip: the [ACS table shells](https://www.census.gov/programs-surveys/acs/technical-documentation/table-shells.html) and the course [Google spreadsheet](https://docs.google.com/spreadsheets/d/1DtGNarbQLaJdtMiINQ7brQ-Y6zawBkkBkp0VGSENsZw/) are useful for finding variable codes._

**Reading response:** _(to be added)_